In [ ]:
!pip install -q \
torch \
transformers \
faiss-cpu \
sentence-transformers \
langchain==0.2.16 \
langchain-community==0.2.16 \
langchain-core==0.2.43 \
langgraph==0.2.5 \
pydantic==1.10.13 \
gradio \
pillow \
tavily-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.6/149.6 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.6 MB/s eta 0:00:00
ERROR: Cannot install langchain-core==0.2.43, langchain==0.2.16 and pydantic==1.10.13 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [ ]:
import io
import numpy as np
import pandas as pd
from PIL import Image

In [ ]:
!pip install faiss-cpu tavily-python

  Using cached faiss_cpu-1.13.2-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.6 kB)
  Using cached tavily_python-0.7.23-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 35.7 MB/s eta 0:00:00


In [ ]:
docs = [
    "RAG stands for Retrieval Augmented Generation.",
    "LangGraph helps build agent-based workflows.",
    "Transformers are used in NLP tasks.",
    "Diabetes is treated using insulin therapy.",
    "Hypertension is controlled with diet and medication.",
    "AI systems can use retrieval to enhance generation."
]

In [ ]:
class InputProcessor:
    def process(self, data):
        try:
            text = data.decode("utf-8")
            return {"type": "text", "text": text, "meta": {}}
        except:
            pass

        try:
            df = pd.read_csv(io.BytesIO(data))
            return {
                "type": "table",
                "text": str(df.head()),
                "meta": {"columns": list(df.columns)}
            }
        except:
            pass

        try:
            img = Image.open(io.BytesIO(data))
            return {
                "type": "image",
                "text": "Image uploaded",
                "meta": {"size": img.size}
            }
        except:
            pass

        return {"type": "unknown", "text": "", "meta": {}}

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

class VectorStore:
    def __init__(self, docs):
        self.docs = docs
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.embeddings = self.model.encode(docs)

        dim = len(self.embeddings[0])
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(np.array(self.embeddings))

    def search(self, query, k=4):
        q_vec = self.model.encode([query])
        D, I = self.index.search(np.array(q_vec), k)
        return [self.docs[i] for i in I[0]]

vector_store = VectorStore(docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from tavily import TavilyClient

class WebRetriever:
    def __init__(self, api_key=None):
        self.client = TavilyClient(api_key="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") if api_key else None

    def search(self, query):
        if not self.client:
            return []
        try:
            res = self.client.search(query=query, max_results=3)
            return [r["content"] for r in res["results"]]
        except:
            return []

In [ ]:
class HybridRetriever:
    def __init__(self):
        self.vector = vector_store
        self.web = WebRetriever(api_key=None)

    def retrieve(self, query):
        docs = self.vector.search(query)

        # Weak context check
        if len(" ".join(docs)) < 50:
            web_docs = self.web.search(query)
            return docs + web_docs, "web"

        return docs, "vector"

In [43]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [44]:
class PlannerAgent:
    def decide(self, query):
        prompt = f"Does this need external knowledge? {query}"

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True).lower()

        if "yes" in response:
            return "web"
        return "vector"


class AnswerAgent:
    def generate(self, query, context):
        prompt = f"""
Answer ONLY from context.
If not found, say NOT FOUND.

Context:
{context}

Question:
{query}

Answer:
"""

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        return response

In [45]:
class Verifier:
    def check(self, query, answer, context):
        if "NOT FOUND" in answer:
            return 0.1

        context_words = set(context.lower().split())
        answer_words = set(answer.lower().split())

        overlap = len(context_words & answer_words)

        score = overlap / (len(answer_words) + 1)

        # Boost if meaningful length
        if len(answer.split()) > 8:
            score += 0.2

        return round(min(score, 0.95), 2)

In [46]:
class AgenticRAG:
    def __init__(self):
        self.retriever = HybridRetriever()
        self.planner = PlannerAgent()
        self.answerer = AnswerAgent()
        self.verifier = Verifier()

    def process(self, query):
        decision = self.planner.decide(query)

        docs, source = self.retriever.retrieve(query)
        context = " ".join(docs)[:500]

        answer = self.answerer.generate(query, context)
        confidence = self.verifier.check(query, answer, context)

        # Corrective RAG
        if confidence < 0.4:
            web_docs = self.retriever.web.search(query)
            context = " ".join(web_docs)
            answer = self.answerer.generate(query, context)
            confidence = self.verifier.check(query, answer, context)
            source = "corrective-web"

        print("Retrieved Docs:", docs)

        return {
            "decision": decision,
            "source": source,
            "retrieved": docs,
            "answer": answer,
            "confidence": confidence
        }

In [47]:
import gradio as gr

rag = AgenticRAG()
processor = InputProcessor()

def rag_app(query, file):
    try:
        extra = ""

        if file is not None:
            data = file.read()
            processed = processor.process(data)
            extra = f"\n📂 File Meta: {processed['meta']}"
            query = processed["text"] or query

        result = rag.process(query)

        return f"""
🧠 Decision: {result['decision']}

🌐 Source: {result['source']}

📚 Retrieved:
{result['retrieved']}

{extra}

✅ Answer:
{result['answer']}

📊 Confidence: {result['confidence']}
"""

    except Exception as e:
        return f"❌ Error: {str(e)}"

ui = gr.Interface(
    fn=rag_app,
    inputs=[
        gr.Textbox(label="Enter Query", lines=2),
        gr.File(label="Upload File (optional)")
    ],
    outputs=gr.Textbox(lines=25, max_lines=50),
    title="🤖 Agentic + Corrective + Multimodal RAG",
    description="Supports text, CSV, image + web fallback"
)

ui.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1b35c5ecff0bd516bc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
